# 01 · Division phenotypes of the mother cells

Each mother cell is turned into a vector: its **cumulative number of divisions** $N(t)$ on a regular time grid,
for example

```
N = 0 0 0 1 1 1 1 2 2 2 2 3 3 3 3 4 4 4
```

All the mothers of all the conditions are then **clustered together** (so the phenotypes are the same for every
condition), drawn as **heatmaps** ordered by the dendrogram, and the notebook reports **how often each phenotype
appears in each condition**.

### How to use it

1. Fill in the cell **Your data** below: one line per folder of Excel files (`*_longitud_madres.xlsx`), with the
   areas you want and the condition they belong to.
2. Run all the cells (**Run › Run All Cells**).
3. The figures (PNG, 200 dpi, sized for slides) and the tables (CSV) are saved in
   `results/01_division_phenotypes/<RUN_NAME>/`.

Nothing else has to be changed. The cells after **Your data** can be read but do not need to be edited.

## Your data

**`EXPERIMENTS`**: one `dict` per folder and condition.

| key | what to write | example |
|---|---|---|
| `folder` | folder with the `<area>-<date>_longitud_madres.xlsx` files (absolute, or relative to this notebook) | `'../November/6/Stacks/Excel'` |
| `condition` | name of the condition; several lines can share one condition (e.g. two experiments of the same treatment) | `'NIT'` |
| `areas` | `'all'`, a range `'1-10'`, or a list `'1-5, 8, 12-14'` | `'11-20'` |
| `flow_start` | time the flow started (`'HH:MM'`), which becomes time 0. If you leave it out (or `None`), time 0 is the **first image of that folder** | `'10:17'` |
| `name` | *(optional)* label of the experiment; by default the date in the file names | `'06/11'` |

**Other settings**

- `RUN_NAME`: name of the folder where this run's results are saved.
- `T_END`: end of the analysis window in minutes since time 0. `None` = the latest time that every condition still has images for.
- `STEP`: spacing of the time grid (min). The vector has one entry every `STEP` minutes.
- `N_PHENOTYPES`: number of phenotypes. `'auto'` = the one with the highest silhouette (section 3); or write a number.
- `CONDITION_COLORS`: *(optional)* your own colours, e.g. `{'LB': '#52514e', 'NIT': '#d9a21b'}`.

In [ ]:
EXPERIMENTS = [
    dict(folder='../Analisis_NIT_GEN/data_gen_nit_bliss/LB_16_10_excel', areas='all',   condition='LB',      flow_start='10:45'),
    dict(folder='../November/6/Stacks/Excel',                         areas='31-40', condition='GEN',     flow_start='10:17'),
    dict(folder='../November/6/Stacks/Excel',                         areas='11-30', condition='NIT',     flow_start='10:17'),
    dict(folder='../November/6/Stacks/Excel',                         areas='1-10',  condition='NIT+GEN', flow_start='10:17'),
]

RUN_NAME = 'NIT_GEN_first_170min'
T_END = 170             # min since time 0, or None
STEP = 10               # min
N_PHENOTYPES = 'auto'   # 'auto' or a number
CONDITION_COLORS = {}   # optional, e.g. {'LB': '#52514e'}

## 1. Reading the Excel files

For each folder, every `*_longitud_madres.xlsx` of the chosen areas is read: the time of each image (row 1) and,
for each channel, the mother's length and, in the row below, the daughter's length in the image where a division
is first seen. That image is where the division is counted.

Mothers that are left out (the table below says how many):

- **empty channel**: no cell in the first 3 images;
- **lost**: the top cell of the channel changed without a division (a comment in the Excel: the tracker lost the
  mother) or the channel emptied, at any time before `T_END`. Their $N(t)$ would freeze and look like an arrest.

In [ ]:
import warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
from IPython.display import display
import mmtools as mm
from mmtools import plots as mp
warnings.filterwarnings('ignore')
mp.set_style()

OUT = Path('results') / '01_division_phenotypes' / RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)

LM, SUMMARY = mm.load_experiments(EXPERIMENTS)
CONDS = list(dict.fromkeys(e['condition'] for e in EXPERIMENTS))       # in the order you wrote them
COL = mp.condition_colors(CONDS, CONDITION_COLORS)
if T_END is None:
    T_END = mm.default_t_end(LM, STEP)
GRID = mm.time_grid(T_END, STEP)
print(f'window: {GRID[0]:g}–{GRID[-1]:g} min, one point every {STEP:g} min ({len(GRID)} points)')
too_short = SUMMARY[SUMMARY.last_image_min < T_END]
if len(too_short):
    print('WARNING: these entries have no images until T_END; their N(t) stays flat at the end:')
    display(too_short)
SUMMARY

In [ ]:
TR = mm.division_vectors(LM, GRID)
TR['use'] = ~TR.empty_start & ~TR.lost
counts = TR.groupby('condition', sort=False).agg(
    channels=('mother', 'size'), areas=('area', 'nunique'),
    empty_channel=('empty_start', 'sum'), lost=('lost', lambda s: int((s & ~TR.loc[s.index, 'empty_start']).sum())),
    mothers_used=('use', 'sum')).reindex(CONDS)
counts['used (%)'] = (100 * counts.mothers_used / (counts.channels - counts.empty_channel)).round(0)
counts

## 2. Each mother as a vector

At every grid point $t$ = `STEP`, 2·`STEP`, …, `T_END`, $N(t)$ is the number of divisions seen in the images up to
$t$. Some mothers, written as words (one digit per grid point):

In [ ]:
ID = TR[TR.use].reset_index(drop=True)
X = np.vstack(ID.N.to_numpy())
rng = np.random.default_rng(0)
ex = pd.concat([ID[ID.condition == c].sample(min(3, (ID.condition == c).sum()), random_state=0) for c in CONDS])
print(f"{'condition':<12}{'exp.':<10}{'area':>5}{'ch.':>5}   N(t), t = {GRID[0]:g} … {GRID[-1]:g} min")
for r in ex.itertuples():
    print(f'{r.condition:<12}{r.experiment:<10}{r.area:>5}{r.channel:>5}   {mm.as_word(r.N)}')

## 3. Clustering

- **Distance** between two mothers: root-mean-square difference of their vectors,
  $d(A,B) = \sqrt{\tfrac{1}{T}\sum_t \big(N_A(t) - N_B(t)\big)^2}$, in divisions ($T$ = number of grid points).
  Two mothers at distance 1 differ, on average, by one division at every time point.
- **Ward linkage** on that distance; the dendrogram orders the rows of the heatmaps.
- **Number of phenotypes $k$**, two criteria:
  - **silhouette**: how much closer each mother is to its own phenotype than to the nearest other one
    (1 = perfectly separated, 0 = on the border);
  - **stability**: the areas of each condition are resampled with replacement, the clustering is repeated and
    compared with the original (median adjusted Rand index, ARI; 1 = identical partition). The shaded band is the
    10th–90th percentile.
- Phenotypes are numbered **P1, P2, …** from the most to the least dividing (by $N$ at the end of the window).

In [ ]:
D, Z = mm.cluster(X)
units = (ID.condition + '|' + ID.experiment.astype(str) + '|' + ID.area.astype(str)).to_numpy()
SEL = mm.choose_k(D, Z, units, ks=range(2, 9))
K = int(SEL.silhouette.idxmax()) if N_PHENOTYPES == 'auto' else int(N_PHENOTYPES)
fig = mp.plot_k(SEL, K, 'Divisions N(t): number of phenotypes'); mp.save(fig, OUT, '01_k_selection'); plt.show()
SEL.round(3)

In [ ]:
PH = mm.numbered_clusters(Z, K, -X[:, -1])          # P1 = the most divisions at the end
ID['phenotype'] = [f'P{j}' for j in PH]
marks = [t for t in np.linspace(GRID[0], GRID[-1], 4)]
marks = [GRID[np.argmin(abs(GRID - t))] for t in marks]
pd.DataFrame({f'P{j}': dict(mothers=int((PH == j).sum()), **{f'mean N at {t:g} min': X[PH == j][:, list(GRID).index(t)].mean() for t in marks})
              for j in range(1, K + 1)}).T.round(2)

## 4. Heatmaps

Rows = mothers, columns = time, colour = number of divisions so far.

- **All the mothers together**: on the left, the dendrogram (coloured below the cut that gives the $k$
  phenotypes); on the right, the phenotype and the condition of each mother.
- **One heatmap per condition**, with the rows in the same order as in the global dendrogram.

In [ ]:
nmax = int(X.max())
cmapN = plt.get_cmap('viridis', nmax + 1); normN = BoundaryNorm(np.arange(-0.5, nmax + 1.5), nmax + 1)
ticks = range(0, nmax + 1, max(1, (nmax + 1) // 10))
fig, ORDER = mp.heatmap_dendrogram(X, GRID, ID.condition.to_numpy(), Z, PH, K, CONDS, COL, cmapN, normN,
                                   'divisions N(t)', 'Divisions of each mother', cbar_ticks=ticks)
mp.save(fig, OUT, '02_heatmap_dendrogram'); plt.show()
fig = mp.heatmaps_by_condition(X, GRID, ID.condition.to_numpy(), ORDER, PH, K, CONDS, COL, cmapN, normN,
                               'divisions N(t)', cbar_ticks=ticks)
mp.save(fig, OUT, '03_heatmaps_by_condition'); plt.show()

## 5. Mean trajectories

Mean $N(t)$ (± SD) of each phenotype, and of each condition.

In [ ]:
fig = mp.mean_trajectories(X, GRID, PH, K, 'divisions N(t)', 'Mean N(t) of each phenotype')
mp.save(fig, OUT, '04_mean_by_phenotype'); plt.show()
fig = mp.mean_by_condition(X, GRID, ID.condition.to_numpy(), CONDS, COL, 'divisions N(t)', 'Mean N(t) of each condition')
mp.save(fig, OUT, '05_mean_by_condition'); plt.show()

## 6. Fraction of each phenotype in each condition

Share of the mothers of each condition in each phenotype. The 95 % intervals come from resampling the **areas**
of each condition (mothers of the same area are not independent).

In [ ]:
FR = mm.fractions(ID, PH, K, CONDS)
fig = mp.plot_fractions(FR, K, CONDS, 'Division phenotypes'); mp.save(fig, OUT, '06_fractions'); plt.show()
fig = mp.plot_fraction_ci(FR, K, CONDS, COL, 'Division phenotypes: fraction in each condition'); mp.save(fig, OUT, '07_fractions_CI'); plt.show()
FR.assign(**{'% (95 % CI)': lambda d: [f'{100*f:.0f} ({100*l:.0f}–{100*h:.0f})' for f, l, h in zip(d.fraction, d.lo, d.hi)]}) \
  .pivot(index='condition', columns='phenotype', values='% (95 % CI)').reindex(CONDS)

## 7. Tables saved

- `mothers.csv`: one row per mother used, with its condition, experiment, area, channel, phenotype and $N(t)$
  (one column per grid point, plus the word).
- `fractions.csv`: fraction of each phenotype in each condition, with its 95 % interval.
- `k_selection.csv`: silhouette and stability for each $k$.
- `settings.txt`: the settings of this run.

In [ ]:
cols = {f'N_{t:g}min': X[:, i] for i, t in enumerate(GRID)}
(ID[['condition', 'experiment', 'area', 'channel', 'phenotype']]
   .assign(word=[mm.as_word(n) for n in X], **cols).to_csv(OUT / 'mothers.csv', index=False))
FR.to_csv(OUT / 'fractions.csv', index=False)
SEL.to_csv(OUT / 'k_selection.csv')
with open(OUT / 'settings.txt', 'w') as f:
    f.write(f'T_END = {T_END}\nSTEP = {STEP}\nN_PHENOTYPES = {N_PHENOTYPES} (used k = {K})\n\n')
    f.write(SUMMARY.to_string(index=False))
print('saved in', OUT.resolve())
sorted(p.name for p in OUT.iterdir())